# Transform Patterns

You have extracted the data. But Seller Alpha uses GBP, Seller Beta uses EUR, and Seller Gamma cannot decide on a date format. Some product names are shouting in ALL CAPS. Others have trailing whitespace. And two products that look absolutely identical somehow survive `drop_duplicates()`.

Before we load anything into the analytics database, we need to **standardise**. That is the "T" in ETL. This is where most of the actual work happens -- and where most of the bugs hide.

In [ ]:
import pandas as pd
import re

## The messy product catalogue

Bazaar has a combined product catalogue that has been assembled from all three sellers. It is, frankly, a mess. Let's load it and see what we are dealing with.

In [ ]:
messy = pd.read_csv("../data/raw_products_messy.csv")
print(f"{len(messy)} rows, {len(messy.columns)} columns")
messy.head(10)

In [ ]:
messy.info()

Already we can see problems. Some product names are uppercase, some lowercase. Currencies appear in mixed case. Let's dig deeper.

In [ ]:
# How many unique currency values?
print("Currency values:")
print(messy["currency"].value_counts())

The same currency appears in different cases: `GBP` and `gbp`, `EUR` and `eur`, `USD` and `usd`. They are the same thing but pandas treats them as different values.

Let's look at the dates.

In [ ]:
# Sample some date values
print("Sample dates:")
print(messy["last_updated"].dropna().sample(10, random_state=42).values)

Three different date formats:

- `01/03/2024` (dd/mm/yyyy -- Alpha's format, British)
- `2024-03-01` (yyyy-mm-dd -- Beta's format, ISO)
- `03/01/2024` (mm/dd/yyyy -- Gamma's format, American)

This is genuinely dangerous. Is `03/01/2024` the 3rd of January or the 1st of March? Without knowing the seller, you cannot tell. We will need to handle this carefully.

## String cleaning

Let's start with the simplest transforms: cleaning up strings.

In [ ]:
# Make a working copy
df = messy.copy()

# Standardise currency to uppercase
df["currency"] = df["currency"].str.upper()
print("Currency after cleaning:")
print(df["currency"].value_counts())

Good -- three clean currency codes. Now categories.

In [ ]:
# Categories: strip whitespace, lowercase
print("Before:")
print(df["category"].value_counts().head(15))

In [ ]:
df["category"] = df["category"].str.strip().str.lower()
print("After:")
print(df["category"].value_counts())

The `.str.strip()` removes leading and trailing whitespace. The `.str.lower()` normalises case. These two operations together fix the vast majority of string inconsistencies.

Now the product names. These need more work.

In [ ]:
# Product names: strip whitespace, title case
df["product_name"] = df["product_name"].str.strip().str.title()

# Check a sample
df["product_name"].sample(10, random_state=42).values

`.str.title()` capitalises the first letter of every word. "VINTAGE LEATHER WALLET" becomes "Vintage Leather Wallet". "handmade ceramic mug" becomes "Handmade Ceramic Mug".

But there is a subtle problem. Some product names have **double spaces**.

In [ ]:
# Find products with double spaces
double_space = df[df["product_name"].str.contains("  ", na=False)]
print(f"Products with double spaces: {len(double_space)}")
double_space["product_name"].head()

In [ ]:
# Fix double (or multiple) spaces with regex
df["product_name"] = df["product_name"].str.replace(r"\s+", " ", regex=True)

# Verify
double_space_after = df[df["product_name"].str.contains("  ", na=False)]
print(f"Products with double spaces after fix: {len(double_space_after)}")

The regex `\s+` matches one or more whitespace characters and replaces them with a single space. This handles double spaces, triple spaces, and any other whitespace oddities.

Or does it?

## The invisible character trap

Let's look at some products near the bottom of the catalogue. Specifically, the ones added on 2024-03-05.

In [ ]:
march_5 = df[df["last_updated"] == "2024-03-05"]
print(f"{len(march_5)} products from March 5th")
march_5[["product_id", "product_name", "price"]]

Look at the product names. "Handmade Ceramic Mug" appears twice. "Vintage Leather Wallet" appears twice. Every product appears to have a duplicate.

Let's try to deduplicate.

In [ ]:
# Try dropping duplicates on product_name and price
deduped = march_5.drop_duplicates(subset=["product_name", "price"])
print(f"Before: {len(march_5)} rows")
print(f"After drop_duplicates: {len(deduped)} rows")

Still 16 rows. `drop_duplicates()` thinks they are all different. But they *look* the same.

Something is wrong. Let's investigate.

In [ ]:
# Compare the first two products character by character
name_1 = march_5.iloc[0]["product_name"]
name_2 = march_5.iloc[1]["product_name"]

print(f"Name 1: '{name_1}'")
print(f"Name 2: '{name_2}'")
print(f"Equal? {name_1 == name_2}")

They look the same. They print the same. But Python says they are not equal.

Time to use `repr()`. This shows the *actual* bytes in the string, including invisible characters.

In [ ]:
print(f"Name 1 repr: {repr(name_1)}")
print(f"Name 2 repr: {repr(name_2)}")

There it is. `\xa0` -- the **non-breaking space**. It is Unicode character U+00A0, and it looks exactly like a regular space on screen. But it is a completely different byte.

One product name has `Handmade Ceramic Mug` (regular space, U+0020). The other has `Handmade\xa0Ceramic Mug` (non-breaking space, U+00A0). Your eyes cannot tell the difference. `drop_duplicates()` can.

This is one of the most common and most frustrating data quality issues. Non-breaking spaces sneak in from:

- Copy-pasting from websites
- Excel exports
- PDFs
- Certain keyboard shortcuts on macOS (Option+Space)

And `\xa0` is just one flavour. There are also zero-width spaces, em spaces, en spaces, and other Unicode whitespace characters. The `.strip()` and regex `\s+` we used earlier *should* catch `\xa0` in many cases, but let's be explicit.

In [ ]:
# Replace non-breaking spaces (and other Unicode whitespace) with regular spaces
df["product_name"] = df["product_name"].str.replace("\xa0", " ", regex=False)

# Re-apply the whitespace normalisation
df["product_name"] = df["product_name"].str.replace(r"\s+", " ", regex=True).str.strip()

# Now check those March 5th products again
march_5_fixed = df[df["last_updated"] == "2024-03-05"]
deduped = march_5_fixed.drop_duplicates(subset=["product_name", "price"])
print(f"Before: {len(march_5_fixed)} rows")
print(f"After drop_duplicates: {len(deduped)} rows")

In [ ]:
# Confirm with repr
name_1 = march_5_fixed.iloc[0]["product_name"]
name_2 = march_5_fixed.iloc[1]["product_name"]
print(f"Name 1 repr: {repr(name_1)}")
print(f"Name 2 repr: {repr(name_2)}")
print(f"Equal? {name_1 == name_2}")

Now they match. The non-breaking space has been replaced with a regular space, and `drop_duplicates()` correctly identifies them as duplicates.

The lesson: when two values look the same but do not compare as equal, **always check `repr()`**. It will reveal invisible characters, trailing whitespace, and encoding issues that your eyes miss.

## Date parsing

Now for the dates. We have three formats mixed together:

| Seller | Format | Example |
|--------|--------|---------|
| Alpha  | dd/mm/yyyy | 01/03/2024 |
| Beta   | yyyy-mm-dd | 2024-03-01 |
| Gamma  | mm/dd/yyyy | 03/01/2024 |

The ISO format (Beta) is unambiguous -- `pd.to_datetime` handles it automatically. But the other two are ambiguous. `01/03/2024` could be January 3rd or March 1st depending on convention.

We need to parse by seller.

In [ ]:
def parse_dates_by_seller(df):
    """
    Parse the last_updated column based on each seller's date format.
    """
    result = df.copy()
    result["last_updated_parsed"] = pd.NaT  # Not a Time (null for datetimes)

    # Alpha: dd/mm/yyyy
    mask_alpha = result["seller"] == "alpha"
    result.loc[mask_alpha, "last_updated_parsed"] = pd.to_datetime(
        result.loc[mask_alpha, "last_updated"], format="%d/%m/%Y", errors="coerce"
    )

    # Beta: yyyy-mm-dd (ISO format)
    mask_beta = result["seller"] == "beta"
    result.loc[mask_beta, "last_updated_parsed"] = pd.to_datetime(
        result.loc[mask_beta, "last_updated"], format="%Y-%m-%d", errors="coerce"
    )

    # Gamma: mm/dd/yyyy
    mask_gamma = result["seller"] == "gamma"
    result.loc[mask_gamma, "last_updated_parsed"] = pd.to_datetime(
        result.loc[mask_gamma, "last_updated"], format="%m/%d/%Y", errors="coerce"
    )

    return result

df = parse_dates_by_seller(df)

# Check for unparsed dates
unparsed = df["last_updated_parsed"].isna().sum()
total_missing_raw = df["last_updated"].isna().sum()
print(f"Unparsed dates: {unparsed} (of which {total_missing_raw} were originally missing)")

In [ ]:
# Show a sample from each seller
for seller in ["alpha", "beta", "gamma"]:
    sample = df[df["seller"] == seller][["last_updated", "last_updated_parsed"]].head(3)
    print(f"\n{seller.upper()}:")
    print(sample.to_string())

The `errors="coerce"` parameter is crucial. If a date string does not match the expected format, it returns `NaT` (Not a Time) instead of raising an error. This lets you identify and investigate the problematic rows instead of crashing the whole pipeline.

We can now drop the original column and rename the parsed one.

In [ ]:
df = df.drop(columns=["last_updated"]).rename(columns={"last_updated_parsed": "last_updated"})
df.dtypes

## Currency conversion

Bazaar's analytics team wants all prices in GBP. We need to convert EUR and USD to GBP.

In a real system, you would pull exchange rates from an API. Here we will use a static lookup.

In [ ]:
# Exchange rates to GBP (approximate, for illustration)
EXCHANGE_RATES = {
    "GBP": 1.0,
    "EUR": 0.86,   # 1 EUR = 0.86 GBP
    "USD": 0.79,   # 1 USD = 0.79 GBP
}

df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["price_gbp"] = df.apply(
    lambda row: round(row["price"] * EXCHANGE_RATES.get(row["currency"], 1.0), 2)
    if pd.notna(row["price"]) else None,
    axis=1
)

df[["product_name", "price", "currency", "price_gbp"]].head(10)

A more efficient approach uses `map` instead of `apply` -- vectorised operations are always faster.

In [ ]:
# Vectorised approach
rate = df["currency"].map(EXCHANGE_RATES)
df["price_gbp"] = (df["price"] * rate).round(2)

# Spot check: a EUR product
eur_sample = df[df["currency"] == "EUR"][["product_name", "price", "currency", "price_gbp"]].head(3)
print(eur_sample)

## Handling missing values

Let's check where the gaps are.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# Look at rows with missing prices
df[df["price"].isna()][["product_id", "product_name", "price", "currency"]]

Missing values are a decision point, not a bug. Your options:

1. **Drop the rows** -- if missing prices make the row useless
2. **Fill with a default** -- if you have a sensible fallback
3. **Flag and keep** -- if downstream consumers need to see them

For the product catalogue, missing prices likely mean the data was not submitted correctly. We will flag them but keep them, so the analytics team can investigate.

In [ ]:
# Add a data quality flag
df["has_missing_data"] = df[["price", "category", "last_updated"]].isna().any(axis=1)
print(f"Rows with missing data: {df['has_missing_data'].sum()}")

## Deduplication

Now that we have cleaned the strings and fixed the invisible characters, we can properly deduplicate.

In [ ]:
print(f"Rows before deduplication: {len(df)}")

# Deduplicate on product_name, price, currency, and seller
# Keep the last occurrence (most recent update)
df_deduped = df.sort_values("last_updated").drop_duplicates(
    subset=["product_name", "price", "currency", "seller"],
    keep="last"
)

print(f"Rows after deduplication: {len(df_deduped)}")
print(f"Removed: {len(df) - len(df_deduped)} duplicates")

Notice we sort by `last_updated` first and keep the last occurrence. This ensures we keep the most recent version of each product when there are duplicates.

## Pure transform functions

Let's pull all of this together into a clean function. A good transform function:

1. Takes a DataFrame in
2. Returns a new DataFrame out
3. Does not modify the input
4. Does not read files, hit databases, or produce side effects

This is a **pure function**. It is easy to test, easy to reason about, and easy to compose with other transforms.

In [ ]:
def clean_seller_data(df, exchange_rates=None):
    """
    Clean and standardise the messy product catalogue.

    Parameters:
        df: Raw product DataFrame
        exchange_rates: Dict mapping currency codes to GBP rates.
                       Defaults to approximate rates.

    Returns:
        Cleaned DataFrame with standardised names, dates, and prices.
    """
    if exchange_rates is None:
        exchange_rates = {"GBP": 1.0, "EUR": 0.86, "USD": 0.79}

    result = df.copy()

    # --- String cleaning ---
    # Replace non-breaking spaces
    result["product_name"] = result["product_name"].str.replace("\xa0", " ", regex=False)
    # Strip, normalise whitespace, title case
    result["product_name"] = (
        result["product_name"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

    # Currency: uppercase
    result["currency"] = result["currency"].str.upper()

    # Category: strip, lowercase
    result["category"] = result["category"].str.strip().str.lower()

    # --- Date parsing ---
    result["last_updated_parsed"] = pd.NaT

    mask_alpha = result["seller"] == "alpha"
    result.loc[mask_alpha, "last_updated_parsed"] = pd.to_datetime(
        result.loc[mask_alpha, "last_updated"], format="%d/%m/%Y", errors="coerce"
    )

    mask_beta = result["seller"] == "beta"
    result.loc[mask_beta, "last_updated_parsed"] = pd.to_datetime(
        result.loc[mask_beta, "last_updated"], format="%Y-%m-%d", errors="coerce"
    )

    mask_gamma = result["seller"] == "gamma"
    result.loc[mask_gamma, "last_updated_parsed"] = pd.to_datetime(
        result.loc[mask_gamma, "last_updated"], format="%m/%d/%Y", errors="coerce"
    )

    result = result.drop(columns=["last_updated"]).rename(
        columns={"last_updated_parsed": "last_updated"}
    )

    # --- Price conversion ---
    result["price"] = pd.to_numeric(result["price"], errors="coerce")
    rate = result["currency"].map(exchange_rates)
    result["price_gbp"] = (result["price"] * rate).round(2)

    # --- Data quality flag ---
    result["has_missing_data"] = result[["price", "category", "last_updated"]].isna().any(axis=1)

    # --- Deduplication ---
    result = result.sort_values("last_updated").drop_duplicates(
        subset=["product_name", "price", "currency", "seller"],
        keep="last"
    )

    return result.reset_index(drop=True)

In [ ]:
# Test it: start from the raw data
clean = clean_seller_data(messy)

print(f"Raw: {len(messy)} rows")
print(f"Clean: {len(clean)} rows")
print(f"Rows with missing data: {clean['has_missing_data'].sum()}")
clean.head()

In [ ]:
# Verify: the original DataFrame is untouched
print(f"Original still has {len(messy)} rows")
print(f"Original columns: {list(messy.columns)}")

The function takes a messy DataFrame and returns a clean one. The original is unchanged. We can call this function from anywhere -- a pipeline, a test, an ad-hoc analysis -- and it will always produce the same output for the same input.

This is the transform pattern you should aim for. Input DataFrame, output DataFrame, no side effects.

## Summary

What we covered:

- **String cleaning** with `.str.strip()`, `.str.lower()`, `.str.title()`, and regex
- **Date parsing** with `pd.to_datetime()` using explicit formats per seller
- **Currency conversion** using a lookup dict and vectorised operations
- **The invisible character trap** -- `\xa0` (non-breaking space) looks identical to a regular space but breaks deduplication. `repr()` reveals the truth.
- **Deduplication** after proper cleaning
- **Missing value handling** with data quality flags
- **Pure transform functions** -- input DataFrame, output DataFrame, no side effects

---

## Exercises

### Exercise 1: Clean the seller CSVs

The daily seller CSV files also need cleaning. Write a function `clean_daily_seller_data(df)` that:

- Strips whitespace from all string columns
- Normalises product names to title case
- Replaces any non-breaking spaces in product names
- Returns the cleaned DataFrame

Test it on the combined seller data from notebook 1.

In [ ]:
# Your code here


### Exercise 2: Validate price ranges

Write a function `validate_prices(df, min_price=0.01, max_price=10000)` that:

- Adds a column `price_valid` that is `True` when the price is between `min_price` and `max_price`, and `False` otherwise (including for missing values)
- Returns the DataFrame with the new column
- Does not modify or remove any rows

In [ ]:
# Your code here


### Exercise 3: Find all invisible characters

Write a function `find_invisible_chars(series)` that takes a pandas Series of strings and returns a DataFrame with columns `index`, `value`, and `repr_value` for any values that contain non-printable or unusual whitespace characters (anything that is not a standard ASCII space, letter, digit, or common punctuation).

Hint: you can use `ord(c)` to get the Unicode code point of a character. Standard printable ASCII is in the range 32-126.

In [ ]:
# Your code here


### Exercise 4: Compose transforms

Write three small, focused transform functions:

1. `normalise_strings(df)` -- handles all string cleaning
2. `convert_currencies(df, rates)` -- handles currency conversion
3. `parse_seller_dates(df)` -- handles date parsing

Then compose them into a single pipeline:

```python
result = parse_seller_dates(convert_currencies(normalise_strings(messy), EXCHANGE_RATES))
```

Verify that the result matches the output of `clean_seller_data(messy)`.

In [ ]:
# Your code here
